In [39]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [40]:
training_set = tf.keras.preprocessing.image_dataset_from_directory(
    "Crop Diseases",
    labels="inferred",
    label_mode="categorical",
    class_names= None,
    color_mode="rgb",
    batch_size=64,
    image_size=(128, 128),
    shuffle=True,
    seed=42,
    validation_split=0.2,
    subset="training",
    interpolation="bilinear",
    follow_links=False,
    crop_to_aspect_ratio=False,
)

Found 13324 files belonging to 17 classes.
Using 10660 files for training.


In [41]:
validation_set = tf.keras.preprocessing.image_dataset_from_directory(
    "Crop Diseases",
    labels="inferred",
    label_mode="categorical",
    class_names= None,
    color_mode="rgb",
    batch_size=64,
    image_size=(128, 128),
    shuffle=True,
    seed=42,
    validation_split=0.2,
    subset="validation",
    interpolation="bilinear",
    follow_links=False,
    crop_to_aspect_ratio=False,
)

Found 13324 files belonging to 17 classes.
Using 2664 files for validation.


In [42]:
AUTOTUNE = tf.data.AUTOTUNE
training_set = training_set.cache().prefetch(buffer_size=AUTOTUNE)
validation_set = validation_set.cache().prefetch(buffer_size=AUTOTUNE)

In [43]:
import ssl
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

In [44]:

ssl._create_default_https_context = ssl._create_unverified_context

# 1. Base model
base_model = MobileNetV2(
    input_shape=(128, 128, 3),
    include_top=False,
    weights='imagenet'
)

# 2. Pipeline with explicit MobileNet scaling [-1, 1]
inputs = tf.keras.Input(shape=(128, 128, 3))

x = layers.RandomFlip("horizontal_and_vertical")(inputs)
x = layers.RandomRotation(0.15)(x)
x = layers.RandomZoom(0.1)(x)

x = preprocess_input(inputs) 
x = base_model(x, training=False)  # Locks BatchNormalization in inference mode
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.5)(x)

outputs = layers.Dense(17, activation='softmax', kernel_regularizer=tf.keras.regularizers.l2(0.01))(x)

model = tf.keras.Model(inputs, outputs)

# 3. Unfreeze
base_model.trainable = True

# 4. Compile with fine-tuning learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


In [45]:
callbacks = [
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=0.000001, verbose=1),
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1)
]

In [46]:

# 5. Fit model
history = model.fit(
    x=training_set,
    validation_data=validation_set,
    epochs=30,
    callbacks=callbacks
)

Epoch 1/30
167/167 ━━━━━━━━━━━━━━━━━━━━ 122s 615ms/step - accuracy: 0.6040 - loss: 1.6927 - val_accuracy: 0.6224 - val_loss: 1.6268 - learning_rate: 3.0000e-05
Epoch 2/30
167/167 ━━━━━━━━━━━━━━━━━━━━ 64s 381ms/step - accuracy: 0.8526 - loss: 0.7609 - val_accuracy: 0.7399 - val_loss: 1.1701 - learning_rate: 3.0000e-05
Epoch 3/30
167/167 ━━━━━━━━━━━━━━━━━━━━ 63s 373ms/step - accuracy: 0.8985 - loss: 0.6118 - val_accuracy: 0.8067 - val_loss: 0.9381 - learning_rate: 3.0000e-05
Epoch 4/30
167/167 ━━━━━━━━━━━━━━━━━━━━ 62s 372ms/step - accuracy: 0.9201 - loss: 0.5384 - val_accuracy: 0.8435 - val_loss: 0.7807 - learning_rate: 3.0000e-05
Epoch 5/30
167/167 ━━━━━━━━━━━━━━━━━━━━ 63s 373ms/step - accuracy: 0.9390 - loss: 0.4890 - val_accuracy: 0.8671 - val_loss: 0.6935 - learning_rate: 3.0000e-05
Epoch 6/30
167/167 ━━━━━━━━━━━━━━━━━━━━ 62s 372ms/step - accuracy: 0.9538 - loss: 0.4442 - val_accuracy: 0.8930 - val_loss: 0.6348 - learning_rate: 3.0000e-05
Epoch 7/30
167/167 ━━━━━━━━━━━━━━━━━━━━ 63s 3